# Fine-tuning de Llama 3.1 8B con LoRA — Asistente de Curso UTZAC

Este notebook cubre las etapas de **preparación de datos**, **ajuste (LoRA)** y
**evaluación** del pipeline. Ejecuta las celdas en orden.

**Requisitos en Colab:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4, gratis).


## 1. Instalación de dependencias

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub

## 2. Autenticación en Hugging Face
Pega tu token (el que generaste con permisos de Write). Necesario para descargar
Llama y, al final, subir tu adaptador LoRA.

In [ ]:
from huggingface_hub import login
login()  # pegará un campo para tu token hf_...

## 3. Preparación de datos

Aquí generas el dataset de fine-tuning: pares pregunta-respuesta basados en TU
temario. Dos formas de obtenerlos:

1. **Manual (recomendado para el curso):** escribe 40-80 pares tú mismo a partir
   de tus notas, en el formato de abajo.
2. **Semi-automático:** pega tus notas en cualquier chat de Claude o ChatGPT y
   pídele "genera 60 pares pregunta-respuesta cortos, en español, basados
   solamente en este texto: [pega tu temario]". Luego revisa y corrige a mano
   — la calidad de este dataset es lo que más impacta el resultado.

Reemplaza la lista `qa_pairs` de abajo con tus datos reales (deja al menos 40).

In [ ]:
qa_pairs = [
    {
        "question": "¿Cómo se evaluan sus asignaturas/materias?",
        "answer": "Se utiliza la metodología de aprendizaje basado en proyectos (ABP)."
    },
    {
        "question": "¿Qué tipo de proyectos se desarrollan en sus materias?",
        "answer": "Se desarrollan proyectos reales que resuelvan algún tipo de problemática sobre nuestro entorno."
    },
    {
        "question": "¿Cómo se compone la calificación final del curso?",
        "answer": "20% por cada avance (son 3), 40% exposición del proyecto final."
    },
    {
        "question": "¿Se puede usar la IA en sus clases?",
        "answer": "Sí se puede usar, pero con un uso adecuado, que les permita generar nuevo conocimiento significativo, con ética y respetando las reglas."
    },
    {
        "question": "¿Qué pasa si en algún avance de proyecto no se obtiene una calificación aprobatoria?",
        "answer": "En caso de no aprobar alguna unidad (avance de proyecto), se tiene derecho a un examen remedial."
    },
    {
        "question": "¿Las asistencias cuentan en su clase?",
        "answer": "Los estudiantes deben asistir a clases por lo menos en un 80%, de lo contrario se les manda a remedial."
    },
    {
        "question": "¿Si no se aprueba la exposición final del proyecto qué consecuencias hay?",
        "answer": "Se tiene derecho a presentar una vez más pero con carácter de extraordinario."
    },
    {
        "question": "¿Los proyectos de sus clases se trabajan en equipo o de manera individual?",
        "answer": "Regularmente se trabajan en equipos de máximo tres integrantes, pero se hacen excepciones para quienes no les gusta trabajar en equipo."
    },
    {
        "question": "¿Cuando un equipo o estudiante va a extraordinario, qué tipo de examen se aplica?",
        "answer": "Se vuelve a presentar el proyecto propuesto desde el inicio del cuatrimestre, el cual debe cumplir con la funcionalidad completa."
    },
    {
        "question": "¿Qué procede cuando no llega a clases después de 15 minutos?",
        "answer": "Dado que desde el inicio del cuatrimestre van a tener que estar desarrollando su proyecto final, deberán estar trabajando el avance que corresponda."
    },
    {
        "question": "¿Qué pasa cuando un equipo termina antes de que se acabe el cuatrimestre?",
        "answer": "Se adelanta la evaluación y pueden dejar de asistir a clases."
    },
    {
        "question": "¿Cómo se lleva a cabo la presentación final de los proyectos?",
        "answer": "Por lo general se realiza en un auditorio con invitados especiales, que les hacen preguntas y los evalúan. Cada equipo tiene solo entre 3 a 5 minutos de exposición."
    },
    {
        "question": "¿En sus clases se estimula el emprendimiento?",
        "answer": "En todas las asignaturas que se imparten por el Dr. Zapata, se les pide a los equipos que desarrollen los proyectos con miras a convertirse en una startup."
    },
    {
        "question": "¿Cómo se desarrollan cada una de las clases?",
        "answer": "Cada una de las clases se desarrolla en un ambiente de respeto; se permiten los comentarios cómicos siempre y cuando no lastimen u ofendan a los demás compañeros."
    },
    {
        "question": "¿Existen los viajes de práctica en sus clases?",
        "answer": "Por lo general todos los cuatrimestres se solicita un viaje de prácticas para visitar empresas de nivel mundial, a fin de motivar a los estudiantes a incorporarse en ese tipo de empresas."
    },
    {
        "question": "¿Cómo se desarrollan las asignaturas?",
        "answer": "Desde el inicio de cada cuatrimestre se les registra en la plataforma Classroom, en la cual desde el primer día de clases se encuentra el material de cada unidad y los entregables con sus fechas."
    },
    {
        "question": "¿Qué plataforma académica se utiliza en sus clases?",
        "answer": "La plataforma empleada en las clases es Classroom de Google."
    },
    {
        "question": "¿Qué pasa cuando se detecta bajo rendimiento de un estudiante?",
        "answer": "En lugar de regañar al estudiante, se le pregunta por su situación personal, familiar, económica y sentimental, para ver cómo se le puede apoyar."
    },
    {
        "question": "¿Hay actividades extracurriculares en sus materias?",
        "answer": "Cada una de las asignaturas se compone de una serie de documentos, audios, videos, prácticas y también de la visualización de películas acordes a la clase y participación en eventos científicos y de emprendimiento."
    },
    {
        "question": "¿Qué pasa cuando se detecta un proyecto con grandes posibilidades de convertirse en una startup?",
        "answer": "Cuando se detecta que un proyecto es bastante factible de convertirse en una startup, se les canaliza al área de emprendimiento de la universidad para que se les asesore en diversos temas (financieros, de marketing, etc.)."
    },
    {
        "question": "¿Han existido proyectos que se hayan convertido en startup?",
        "answer": "Sí, existen varias startups que han surgido de proyectos académicos."
    },
    # Puedes seguir agregando más pares aquí si quieres mejorar el ajuste.
]

print(f"Total de ejemplos: {len(qa_pairs)}")
assert len(qa_pairs) >= 10, "Agrega más ejemplos antes de continuar (idealmente 40+)."

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Eres el asistente del curso de un profesor de la Universidad Tecnológica de "
    "Zacatecas (UTZAC). Responde de forma breve, clara y en español."
)

def format_example(example):
    text = (
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{example['question']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{example['answer']}<|eot_id|>"
    )
    return {"text": text}

dataset = Dataset.from_list(qa_pairs).map(format_example)
dataset = dataset.train_test_split(test_size=0.15, seed=42)
dataset

## 4. Carga del modelo base en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "NousResearch/Meta-Llama-3.1-8B-Instruct"  # copia sin gating, mismos pesos

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

## 5. Configuración de LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Entrenamiento (fine-tuning)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./llama-utzac-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

## 7. Evaluación de salidas

Compara respuestas del modelo base vs. el modelo ajustado, sobre preguntas que
NO estaban en el set de entrenamiento (usa el split de test).

In [ ]:
def generate_answer(question, max_new_tokens=150):
    prompt = (
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_text.split(prompt.replace("<|start_header_id|>", "").replace("<|end_header_id|>", ""))[-1].strip()

for example in dataset["test"]:
    print("PREGUNTA:", example["question"])
    print("ESPERADA:", example["answer"])
    print("MODELO  :", generate_answer(example["question"]))
    print("-" * 60)

## 8. Guardar y subir el adaptador a Hugging Face Hub

Esto deja evidencia pública de la etapa de fine-tuning, aunque el endpoint de
producción (FastAPI) use Groq para servir las respuestas — ver README del
proyecto para la justificación de esta arquitectura.

In [ ]:
HF_USERNAME = "tu-usuario-hf"  # <-- cámbialo
REPO_NAME = "llama-utzac-curso-lora"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
print(f"Adaptador subido a: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")